In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from scipy.interpolate import interp1d

train = pd.read_csv(
    "/content/drive/MyDrive/IV_Surface/data/raw/dataset.csv"
)

print(train.shape)

Mounted at /content/drive
(975, 30)


In [ ]:
def create_validation_copy(df, seed=42):

    rng = np.random.default_rng(seed)

    df_copy = df.copy()

    hidden_positions = []

    option_cols = df.columns[2:]

    for row_idx in range(len(df)):

        row = df.loc[row_idx, option_cols]

        known = np.where(~row.isna())[0]

        if len(known) < 10:
            continue

        hide = rng.choice(
            known,
            size=max(1, int(len(known) * 0.20)),
            replace=False
        )

        for pos in hide:

            hidden_positions.append(
                (
                    row_idx,
                    option_cols[pos],
                    row.iloc[pos]
                )
            )

            df_copy.loc[row_idx, option_cols[pos]] = np.nan

    return df_copy, hidden_positions


val_df, hidden_positions = create_validation_copy(train)

print("Hidden values:", len(hidden_positions))

Hidden values: 3958


In [ ]:
strike_errors = []

option_cols = train.columns[2:]

for row_idx, col, true_val in hidden_positions:

    row = val_df.loc[row_idx, option_cols].values.astype(float)

    x = np.arange(len(row))

    valid = ~np.isnan(row)

    if valid.sum() < 2:
        continue

    f = interp1d(
        x[valid],
        row[valid],
        kind="linear",
        fill_value="extrapolate"
    )

    target_idx = option_cols.get_loc(col)

    pred = float(f(target_idx))

    strike_errors.append({
        "column": col,
        "error": (pred - true_val) ** 2
    })

print("Records:", len(strike_errors))

Records: 3958


In [ ]:
err_df = pd.DataFrame(strike_errors)

summary = (
    err_df
    .groupby("column")["error"]
    .mean()
    .sort_values(ascending=False)
)

summary

,error
column,
NIFTY27JAN2623800PE,0.002251
NIFTY27JAN2626500CE,0.001554
NIFTY27JAN2623900PE,0.000348
NIFTY27JAN2625200CE,0.000282
NIFTY27JAN2626400CE,0.000282
NIFTY27JAN2626300CE,0.000109
NIFTY27JAN2625700CE,0.000102
NIFTY27JAN2625800CE,0.000092
NIFTY27JAN2625300CE,0.000084


In [ ]:
print(summary)

column
NIFTY27JAN2623800PE    0.002251
NIFTY27JAN2626500CE    0.001554
NIFTY27JAN2623900PE    0.000348
NIFTY27JAN2625200CE    0.000282
NIFTY27JAN2626400CE    0.000282
NIFTY27JAN2626300CE    0.000109
NIFTY27JAN2625700CE    0.000102
NIFTY27JAN2625800CE    0.000092
NIFTY27JAN2625300CE    0.000084
NIFTY27JAN2624000PE    0.000063
NIFTY27JAN2625400CE    0.000053
NIFTY27JAN2625100PE    0.000038
NIFTY27JAN2626200CE    0.000035
NIFTY27JAN2625500CE    0.000031
NIFTY27JAN2625900CE    0.000021
NIFTY27JAN2626100CE    0.000020
NIFTY27JAN2624600PE    0.000017
NIFTY27JAN2624800PE    0.000017
NIFTY27JAN2624100PE    0.000015
NIFTY27JAN2624900PE    0.000014
NIFTY27JAN2625600CE    0.000009
NIFTY27JAN2624700PE    0.000009
NIFTY27JAN2624200PE    0.000009
NIFTY27JAN2624500PE    0.000008
NIFTY27JAN2625000PE    0.000006
NIFTY27JAN2624300PE    0.000004
NIFTY27JAN2624400PE    0.000003
NIFTY27JAN2626000CE    0.000002
Name: error, dtype: float64


In [6]:
summary.head(8)

,error
column,
NIFTY27JAN2623800PE,0.002251
NIFTY27JAN2626500CE,0.001554
NIFTY27JAN2623900PE,0.000348
NIFTY27JAN2625200CE,0.000282
NIFTY27JAN2626400CE,0.000282
NIFTY27JAN2626300CE,0.000109
NIFTY27JAN2625700CE,0.000102
NIFTY27JAN2625800CE,0.000092


In [7]:
edge_cols = [
    "NIFTY27JAN2623800PE",
    "NIFTY27JAN2623900PE",
    "NIFTY27JAN2626400CE",
    "NIFTY27JAN2626500CE"
]

for col in edge_cols:
    pct = train[col].isna().mean() * 100
    print(f"{col}: {pct:.2f}% missing")

NIFTY27JAN2623800PE: 20.82% missing
NIFTY27JAN2623900PE: 20.62% missing
NIFTY27JAN2626400CE: 21.85% missing
NIFTY27JAN2626500CE: 18.87% missing


In [8]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

sandbox = pd.read_csv(
    "/content/drive/MyDrive/IV_Surface/data/raw/sandbox_solution.csv"
)

print(sandbox.shape)
print(sandbox.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(5460, 2)
                                      id     value
0  07-01-2026 09:15||NIFTY27JAN2624100PE  0.143129
1  07-01-2026 09:15||NIFTY27JAN2625500CE  0.102447
2  07-01-2026 09:15||NIFTY27JAN2625800CE  0.102447
3  07-01-2026 09:20||NIFTY27JAN2624000PE  0.147267
4  07-01-2026 09:20||NIFTY27JAN2624200PE  0.147267
